# 01b — Payment outcomes for net-return analysis

## Purpose
Notebook 01 saved a deliberately small repayment table (`loss_outcomes_2011_2015_36m.parquet`) with principal received, recoveries and recovery fees. That is enough for a principal-loss measure, but not for a cash-return measure: it has no interest, late fees or total payments. Judging an approval policy by principal loss alone ignores the interest earned on loans the policy would have rejected.

This notebook reads the additional repayment fields from the untouched raw file for exactly the same 603,589 loans and saves them as a **separate** table. It does not modify the PD table, the loss table, the feature module, or any model checkpoint.

## Input and scope
* `data/raw/accepted_2007_to_2018Q4.csv.gz` (read-only)
* `data/processed/pd_cohort_2011_2015_36m.parquet` (defines the loan IDs and row order)
* `data/processed/loss_outcomes_2011_2015_36m.parquet` (used only as a consistency check)
* Output: `data/processed/payment_outcomes_2011_2015_36m.parquet`

These fields are post-origination outcomes. They must never be used as model inputs or as features in any policy decision. They exist only to evaluate decisions retrospectively.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve().parent

raw_file = project_root / "data" / "raw" / "accepted_2007_to_2018Q4.csv.gz"
pd_file = project_root / "data" / "processed" / "pd_cohort_2011_2015_36m.parquet"
loss_file = project_root / "data" / "processed" / "loss_outcomes_2011_2015_36m.parquet"
payment_file = (
    project_root / "data" / "processed" / "payment_outcomes_2011_2015_36m.parquet"
)

for path in (raw_file, pd_file, loss_file):
    if not path.is_file():
        raise FileNotFoundError(path)

if payment_file.exists():
    raise FileExistsError(f"Refusing to overwrite: {payment_file}")

cohort = pd.read_parquet(pd_file, columns=["id", "issue_month", "charged_off"])
cohort["id"] = cohort["id"].astype("string")

assert cohort["id"].is_unique
assert len(cohort) == 603_589

cohort_id_set = set(cohort["id"])

print("Cohort loans:", f"{len(cohort):,}")
print("Charged off: ", f"{int(cohort['charged_off'].sum()):,}")
print("Output file:  ", payment_file.name)

Cohort loans: 603,589
Charged off:  84,227
Output file:   payment_outcomes_2011_2015_36m.parquet


## Read the repayment fields for the cohort IDs
The raw file is read in chunks and filtered to the cohort IDs. IDs are read as strings because the raw column mixes numeric-looking and text values (the same reason Notebook 01 stores `id` as a string).

In [2]:
raw_columns = [
    "id",
    "loan_status",
    "funded_amnt",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
]

parts = []
rows_scanned = 0

for chunk in pd.read_csv(
    raw_file,
    usecols=raw_columns,
    dtype={"id": "string"},
    chunksize=200_000,
    compression="infer",
    low_memory=False,
):
    rows_scanned += len(chunk)
    selected = chunk.loc[chunk["id"].isin(cohort_id_set)]
    parts.append(selected.copy())

payments_raw = pd.concat(parts, ignore_index=True)

print("Raw rows scanned:", f"{rows_scanned:,}")
print("Rows matched to cohort:", f"{len(payments_raw):,}")

assert payments_raw["id"].is_unique, "Duplicate IDs in the raw payment fields."
assert set(payments_raw["id"]) == cohort_id_set, (
    "Raw file and PD cohort do not contain the same loan IDs."
)

Raw rows scanned: 2,260,701
Rows matched to cohort: 603,589


## Align to the PD table and verify
The payment table is aligned to the PD table's row order. Three checks follow: the status in the raw file agrees with the saved target, the money fields are complete, and the fields that Notebook 01 already saved match exactly.

In [5]:
money_columns = [
    "funded_amnt",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
]

aligned = cohort[["id", "charged_off"]].merge(
    payments_raw,
    on="id",
    how="left",
    validate="one_to_one",
)

assert len(aligned) == len(cohort)
assert aligned["id"].equals(cohort["id"]), "Row order differs from the PD table."

# 1. Raw status agrees with the target saved by Notebook 01.
status_is_charged_off = aligned["loan_status"].eq("Charged Off").astype("int8")
assert (status_is_charged_off == aligned["charged_off"]).all()
assert aligned["loan_status"].isin(["Fully Paid", "Charged Off"]).all()

# 2. Money fields are complete and handle negative accounting artifacts.
missing_money = aligned[money_columns].isna().sum()
print("Missing values in money fields:")
print(missing_money.to_string())
assert missing_money.sum() == 0

negative_counts = (aligned[money_columns] < 0).sum()
if negative_counts.sum() > 0:
    print("\nNegative values found (accounting adjustments). Clipping to 0:")
    print(negative_counts[negative_counts > 0].to_string())
    aligned[money_columns] = aligned[money_columns].clip(lower=0)

assert (aligned[money_columns] >= 0).all().all()

# 3. Overlap with the existing loss table matches exactly.
loss_check = pd.read_parquet(
    loss_file,
    columns=[
        "id",
        "funded_amnt",
        "total_rec_prncp",
        "recoveries",
        "collection_recovery_fee",
    ],
)
loss_check["id"] = loss_check["id"].astype("string")
loss_check = loss_check.rename(
    columns={
        c: f"{c}_loss"
        for c in loss_check.columns
        if c != "id"
    }
)

compare = aligned.merge(loss_check, on="id", how="left", validate="one_to_one")
assert compare["funded_amnt_loss"].notna().all()

for column in [
    "funded_amnt",
    "total_rec_prncp",
    "recoveries",
    "collection_recovery_fee",
]:
    assert np.allclose(compare[column], compare[f"{column}_loss"], atol=1e-6), column

print("\nStatus, completeness and loss-table consistency checks passed.")
print("Missing last_pymnt_d:", int(aligned["last_pymnt_d"].isna().sum()))

Missing values in money fields:
funded_amnt                0
total_pymnt                0
total_rec_prncp            0
total_rec_int              0
total_rec_late_fee         0
recoveries                 0
collection_recovery_fee    0

Negative values found (accounting adjustments). Clipping to 0:
total_rec_late_fee    3

Status, completeness and loss-table consistency checks passed.
Missing last_pymnt_d: 448


## Reconciliation audit: what does `total_pymnt` contain?
Before defining a net-return measure, check how `total_pymnt` relates to its components. The question is whether recoveries on charged-off loans are already inside `total_pymnt`. If they are, adding recoveries again would double count. This cell **reports** the relationship; it does not assert one, because the answer decides the formula used in Notebook 05b.

In [6]:
components = (
    aligned["total_rec_prncp"]
    + aligned["total_rec_int"]
    + aligned["total_rec_late_fee"]
)
residual = aligned["total_pymnt"] - components
tolerance = 0.02

summary_rows = []
for label, mask in [
    ("Fully Paid", aligned["charged_off"].eq(0)),
    ("Charged Off", aligned["charged_off"].eq(1)),
]:
    r = residual[mask]
    rec = aligned.loc[mask, "recoveries"]
    summary_rows.append(
        {
            "group": label,
            "loans": int(mask.sum()),
            "residual_is_zero": float((r.abs() <= tolerance).mean()),
            "residual_equals_recoveries": float(
                ((r - rec).abs() <= tolerance).mean()
            ),
            "mean_residual": float(r.mean()),
            "mean_recoveries": float(rec.mean()),
        }
    )

audit = pd.DataFrame(summary_rows).set_index("group")
print(audit.round(4).to_string())
print()
print(
    "Reading guide: if 'residual_equals_recoveries' is near 1 for charged-off\n"
    "loans, recoveries are already part of total_pymnt. If 'residual_is_zero'\n"
    "is near 1, they are not."
)

              loans  residual_is_zero  residual_equals_recoveries  mean_residual  mean_recoveries
group                                                                                            
Fully Paid   519362            1.0000                         1.0         0.0000           0.0000
Charged Off   84227            0.2184                         1.0       914.1927         914.1927

Reading guide: if 'residual_equals_recoveries' is near 1 for charged-off
loans, recoveries are already part of total_pymnt. If 'residual_is_zero'
is near 1, they are not.


**Record of Result:**
The audit above confirms that `total_pymnt` includes recoveries, as the mathematical residual exactly equals the recorded recoveries for 100% of the charged-off loans. To avoid double-counting, Notebook 05b will use the following net-cash formula:

`net_cash = total_pymnt - collection_recovery_fee - funded_amnt`

Net return on funded amount is `net_cash / funded_amnt`. This is an undiscounted, cash-basis measure. It ignores funding cost, servicing cost, and the time value of money, so it represents a return on funded principal, not a comprehensive profit figure.

## Sanity view by issue year

In [7]:
by_year = aligned.assign(
    issue_year=cohort["issue_month"].dt.year.to_numpy()
)

view = (
    by_year.groupby(["issue_year", "charged_off"])
    .agg(
        loans=("id", "size"),
        funded=("funded_amnt", "sum"),
        total_pymnt=("total_pymnt", "sum"),
        interest=("total_rec_int", "sum"),
        recoveries=("recoveries", "sum"),
    )
    .assign(cash_per_dollar_funded=lambda x: x["total_pymnt"] / x["funded"])
)
print(view.round(3).to_string())

                         loans        funded   total_pymnt      interest   recoveries  cash_per_dollar_funded
issue_year charged_off                                                                                       
2011       0             12602  1.198667e+08  1.373045e+08  1.743064e+07         0.00                   1.145
           1              1499  1.266440e+07  7.537065e+06  1.627518e+06    587693.66                   0.595
2012       0             37567  4.393890e+08  5.169686e+08  7.756005e+07         0.00                   1.177
           1              5903  6.828220e+07  4.241702e+07  1.018503e+07   3928548.36                   0.621
2013       0             88044  1.121836e+09  1.313943e+09  1.920657e+08         0.00                   1.171
           1             12378  1.502542e+08  9.873358e+07  2.347414e+07  11153460.46                   0.657
2014       0            140255  1.775140e+09  2.052464e+09  2.772383e+08         0.00                   1.156
          

## Save and verify
The saved table keeps the loan ID, the recorded final status and the repayment fields only. It contains no model inputs. Join it to other tables on `id`.

In [8]:
payment_columns = ["id"] + [c for c in raw_columns if c != "id"]
payment_outcomes = aligned[payment_columns].copy()
payment_outcomes["id"] = payment_outcomes["id"].astype("string")

assert payment_outcomes["id"].is_unique
assert len(payment_outcomes) == 603_589
assert payment_outcomes["id"].equals(cohort["id"])

if payment_file.exists():
    raise FileExistsError(f"Refusing to overwrite: {payment_file}")

payment_outcomes.to_parquet(payment_file, index=False, compression="snappy")

check = pd.read_parquet(payment_file)
assert check.shape == payment_outcomes.shape
assert check["id"].equals(cohort["id"])
assert np.isclose(check["total_pymnt"].sum(), payment_outcomes["total_pymnt"].sum())

print(f"Saved and verified: {payment_file.name}")
print(f"Size: {payment_file.stat().st_size / (1024**2):.1f} MiB")
print("Shape:", check.shape)
print("Columns:", list(check.columns))

Saved and verified: payment_outcomes_2011_2015_36m.parquet
Size: 15.2 MiB
Shape: (603589, 10)
Columns: ['id', 'loan_status', 'funded_amnt', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d']


## Conclusion and handoff
This notebook added `payment_outcomes_2011_2015_36m.parquet`, a repayment-outcome table for the same 603,589 loans as the PD table, in the same row order. Its overlap with the Notebook 01 loss table was verified exactly.

**What it enables:** Notebook 05b can compare approval policies by net cash return as well as by principal loss, and can ask whether the loans a policy rejects were actually unprofitable once interest is counted.

**What it does not do:** It contains no model inputs, and the net-cash measure is undiscounted and excludes funding and servicing costs. It describes originated loans only.

**Note:**
The data preparation, split design, and feature pipeline are finished. The design choices raised earlier (keeping `verification_status`, `home_ownership`, and `addr_state` as inputs; the label-availability assumption) are not reasons to redo Notebooks 01–03. I have documented them as limitations and will test them later as sensitivity models in new notebooks. A sensitivity model means a new feature list and a new preprocessor on top of `credit_features.py`. It doesn't change the existing artifacts.

**Next:** `04b-Model-Diagnostics-and-Drift.ipynb`, then Notebook 05b using this table.